# Modelling and Tracking Shared E-Scooters under Correlated Noise in Urban Environments

## Part I. Effects of Ignoring Correlated State and Measurement Noise in Kalman Filtering
### Background and useful recommendations

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

![image.png](attachment:image.png)

In [ ]:
# defining the constants

VARIANCE_A = 0.2**2
VARIANCE_ETA = 20**2
SIGMA_A = np.sqrt(VARIANCE_A)
SIGMA_ETA = np.sqrt(VARIANCE_ETA)

POINTS = 200
M = 500

T = 1
X_INIT = 5
V_INIT = 1

LAMBDA_CORRELATED = 0.1
LAMBDA_UNCORRELATED = 1000

![image.png](attachment:image.png)

In [ ]:
# state-space model for the scooter's motion

# transition matrix Phi
PHI = np.array([[1, T],
                [0, 1]])

# input matrix G that defines how the random acceleration a_i influences the state vector
G = np.array([[T**2 / 2],
              [T]])

# observation matrix H that relates the measurement vector to the state vector 
H = np.array([[1, 0]])

# covariance matrix Q of the state noise G * a_i
Q = G @ G.T * VARIANCE_A

print(Q)

# covariance matrix R of the measurement noise
R = np.array([[VARIANCE_ETA]])

print(R)

[[0.01 0.02]
 [0.02 0.04]]
[[400]]


In [ ]:
# setting the initial conditions for the filtered estimate and the filtration error covariance matrix

# initial filtered estimate of the state vector 
X_0_0 = np.array([[X_INIT],
                  [V_INIT]])

# initial filtration error covariance matrix,
# identical to the more accurate matrix from assignment 6
P_0_0 = np.array([[100, 0],
                  [0, 100]])

In [ ]:
# Kalman filter as a function,
# the dimensions are taken from the matrices, so it works both for X_i = [x, V] and for X_i = [x, V, a]
def kalman_filter(measurements, phi, h, q, r, x_0_0, p_0_0):
    size = measurements.size
    n = phi.shape[0]   # dimension of the state vector
    m = h.shape[0]     # dimension of the measurement vector

    x_pred = np.zeros((size, n, 1))   # predicted state X_{i,i-1}
    p_pred = np.zeros((size, n, n))   # prediction error covariance P_{i,i-1}
    x_filt = np.zeros((size, n, 1))   # filtered state X_{i,i}
    p_filt = np.zeros((size, n, n))   # filtration error covariance P_{i,i}
    k = np.zeros((size, n, m))        # filter gain K_i

    identity = np.eye(n)
    x_prev = x_0_0
    p_prev = p_0_0

    x_pred[0] = x_prev
    p_pred[0] = p_prev
    x_filt[0] = x_pred[0]
    p_filt[0] = p_pred[0]

    for i in range(1, size):
        # prediction
        x_pred[i] = phi @ x_prev
        p_pred[i] = phi @ p_prev @ phi.T + q

        # filtration
        k[i] = p_pred[i] @ h.T @ np.linalg.inv(h @ p_pred[i] @ h.T + r)
        x_filt[i] = x_pred[i] + k[i] @ (measurements[i] - h @ x_pred[i])
        p_filt[i] = (identity - k[i] @ h) @ p_pred[i]

        x_prev = x_filt[i]
        p_prev = p_filt[i]

    return x_pred, p_pred, x_filt, p_filt, k

![image.png](attachment:image.png)

In [ ]:
# find uncorrelated noise zeta
def get_zeta(lam, variance_a=VARIANCE_A, t=T):
    variance_zeta = variance_a * (1 - np.exp(-2 * lam * t))
    return np.random.normal(loc=0.0, scale=np.sqrt(variance_zeta))

![image.png](attachment:image.png)
![image-2.png](attachment:image-2.png)

In [ ]:
# a function to generate a true trajectory of an e-scooter,
# it also returns the true velocity and the true correlated acceleration a_i
def generate_true_trajectory(lam, size=POINTS, init_coord=X_INIT, init_velocity=V_INIT, sigma_a=SIGMA_A, t=T):
    traj_points = np.zeros(size)
    velocity_points = np.zeros(size)
    acceleration_points = np.zeros(size)
    traj_points[0] = init_coord
    velocity_points[0] = init_velocity
    acceleration_points[0] = np.random.normal(loc=0.0, scale=sigma_a)

    for i in range(1, size):
        traj_points[i] = traj_points[i-1] + velocity_points[i-1] * t + acceleration_points[i-1] * t**2 / 2
        velocity_points[i] = velocity_points[i-1] + acceleration_points[i-1] * t
        zeta = get_zeta(lam, sigma_a**2, t)
        acceleration_points[i] = np.exp(-lam * t) * acceleration_points[i-1] + zeta

    return traj_points, velocity_points, acceleration_points

![image.png](attachment:image.png)

In [ ]:
# find uncorrelated noise final sigma
def get_final_sigma(lam, variance_eta=VARIANCE_ETA, t=T):
    variance_fsigma = variance_eta * (1 - np.exp(-2 * lam * t))
    return np.random.normal(loc=0.0, scale=np.sqrt(variance_fsigma))

# a function to generate measurements of the coordinate with the noise eta_i from equation (3)
def generate_measurements(true_trajectory, lam, t=T, sigma_eta=SIGMA_ETA):
    measurements = np.zeros(len(true_trajectory))
    eta = np.random.normal(loc=0.0, scale=sigma_eta)

    for i in range(len(true_trajectory)):
        eta = np.exp(-lam * t) * eta + get_final_sigma(lam, sigma_eta**2, t)
        measurements[i] = true_trajectory[i] + eta
    return measurements

Examples of the generated noise: uncorrelated ($\lambda = 1000$) and correlated ($\lambda = 0.1$)

In [ ]:
# realizations of the acceleration a_i and the measurement noise eta_i for both values of lambda
steps = np.arange(POINTS * T, step = T)

_, _, acc_uncorr = generate_true_trajectory(LAMBDA_UNCORRELATED)
_, _, acc_corr = generate_true_trajectory(LAMBDA_CORRELATED)

# measurements of a zero trajectory are the measurement noise itself
eta_uncorr = generate_measurements(np.zeros(POINTS), LAMBDA_UNCORRELATED)
eta_corr = generate_measurements(np.zeros(POINTS), LAMBDA_CORRELATED)

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(steps, acc_uncorr, color="gray", linewidth=1, label=f"Uncorrelated, $\\lambda$ = {LAMBDA_UNCORRELATED}")
axes[0].plot(steps, acc_corr, color="blue", linewidth=1.5, label=f"Correlated, $\\lambda$ = {LAMBDA_CORRELATED}")
axes[0].set_title("Random Acceleration $a_i$")
axes[0].set_xlabel("Time, s")
axes[0].set_ylabel("Acceleration $a_i$, m/s$^2$")
axes[0].grid(True, color="black", linewidth=0.3)
axes[0].legend(frameon=True)

axes[1].plot(steps, eta_uncorr, color="gray", linewidth=1, label=f"Uncorrelated, $\\lambda$ = {LAMBDA_UNCORRELATED}")
axes[1].plot(steps, eta_corr, color="red", linewidth=1.5, label=f"Correlated, $\\lambda$ = {LAMBDA_CORRELATED}")
axes[1].set_title("Measurement Noise $\\eta_i$")
axes[1].set_xlabel("Time, s")
axes[1].set_ylabel("Measurement noise $\\eta_i$, m")
axes[1].grid(True, color="black", linewidth=0.3)
axes[1].legend(frameon=True)

fig.tight_layout()
plt.show()

### Functions for plotting and for the statistical analysis over M runs

In [ ]:
# a function to plot a true trajectory, measured points and the filtered estimate (from assignment 6)
def plot_filter_results(true_traj, measured_traj, estimated_traj, title, step_size=T):
    steps = np.arange(true_traj.size * step_size, step = step_size)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(steps, true_traj, color="green", linewidth=1.5, label="True Trajectory $x_i$")
    ax.scatter(steps, measured_traj, color="red", s=6, alpha=0.6, label="Measurements $z_i = x_i + \\eta_i$")
    ax.plot(steps, estimated_traj, color="blue", linewidth=1.5, label="Kalman Filter Estimate $x_{i,i}$")
    ax.set_title(title)
    ax.set_xlabel("Time, s")
    ax.set_ylabel("Coordinate x, m")
    ax.grid(True, color="black", linewidth=0.3)
    ax.legend(frameon=True)
    fig.tight_layout()
    plt.show()

# a function to plot several error curves on one figure (from assignment 6)
def plot_errors(curves, title, step_size=T, start=2, sigma_eta=None, ylabel="Estimation error, m", legend_outside=False):
    # curves: list of (values, label, color)
    steps = np.arange(len(curves[0][0]) * step_size, step = step_size)
    fig, ax = plt.subplots(figsize=(12, 5))
    for values, label, color in curves:
        ax.plot(steps[start:], values[start:], color=color, linewidth=1.5, label=label)
    if sigma_eta is not None:
        ax.axhline(sigma_eta, color="red", linestyle="--", linewidth=1, label="Measurement error $\\sigma_\\eta$")
    ax.set_title(title)
    ax.set_xlabel("Time, s")
    ax.set_ylabel(ylabel)
    ax.grid(True, color="black", linewidth=0.3)
    # many curves: the legend is placed to the right of the plot so that it does not cover them
    if legend_outside:
        ax.legend(frameon=True, loc="upper left", bbox_to_anchor=(1.01, 1))
    else:
        ax.legend(frameon=True)
    fig.tight_layout()
    plt.show()

In [ ]:
# one realization: true trajectory, measurements and the filtered estimate of the coordinate
def run_single(lam_a, lam_eta, title):
    true_traj, _, _ = generate_true_trajectory(lam_a)
    measurements = generate_measurements(true_traj, lam_eta)
    _, _, x_filt, _, _ = kalman_filter(measurements, PHI, H, Q, R, X_0_0, P_0_0)
    plot_filter_results(true_traj, measurements, x_filt[:, 0, 0], title)

# true and calculated errors of the Kalman filter over M runs
# lam_a, lam_eta: used to generate the true trajectories and measurements; phi, h, q, r, x_0_0, p_0_0: used inside the filter
def run_experiment(lam_a, lam_eta, phi, h, q, r, x_0_0, p_0_0, m_runs=M):
    n = phi.shape[0]
    sq_err_filt = np.zeros((m_runs, POINTS, n))   # (X_i - X_{i,i})^2 for every component of the state vector
    sq_err_pred = np.zeros((m_runs, POINTS, n))   # (X_i - X_{i,i-1})^2

    for run in range(m_runs):
        # new true trajectory and new measurements in every run
        true_traj, true_vel, true_acc = generate_true_trajectory(lam_a)
        measurements = generate_measurements(true_traj, lam_eta)
        x_pred, p_pred, x_filt, p_filt, _ = kalman_filter(measurements, phi, h, q, r, x_0_0, p_0_0)

        # the filter with n = 2 estimates [x, V], the filter with n = 3 estimates [x, V, a]
        true_state = np.column_stack((true_traj, true_vel, true_acc))[:, :n]
        sq_err_filt[run] = (true_state - x_filt[:, :, 0])**2
        sq_err_pred[run] = (true_state - x_pred[:, :, 0])**2

    # root of the sum over runs divided by (M - 1), as in assignment 6
    true_err_filt = np.sqrt(np.sum(sq_err_filt, axis=0) / (m_runs - 1))
    true_err_pred = np.sqrt(np.sum(sq_err_pred, axis=0) / (m_runs - 1))

    # calculated errors: square roots of the diagonal elements of P_{i,i} and P_{i,i-1},
    # they do not depend on the measurements, so the matrices from the last run are used
    calc_err_filt = np.sqrt(np.diagonal(p_filt, axis1=1, axis2=2))
    calc_err_pred = np.sqrt(np.diagonal(p_pred, axis1=1, axis2=2))
    return true_err_filt, true_err_pred, calc_err_filt, calc_err_pred

# steady-state values: mean over the last 100 steps
def print_steady_errors(true_err, calc_err, name, unit="m"):
    steady_true = np.mean(true_err[-100:])
    steady_calc = np.mean(calc_err[-100:])
    print(f"{name}: true error = {steady_true:.3f} {unit}, calculated error = {steady_calc:.3f} {unit}, "
          f"true / calculated = {steady_true / steady_calc:.2f}")
    return steady_true, steady_calc

### Scenario 1. Optimal Kalman filter for uncorrelated state and measurement noise
Baseline case: both process and measurement noises are white and independent ($\lambda = 1000$ for $a_i$ and $\eta_i$).

In [ ]:
# Scenario 1: uncorrelated acceleration and uncorrelated measurement noise
run_single(LAMBDA_UNCORRELATED, LAMBDA_UNCORRELATED,
           "Scenario 1: Uncorrelated Acceleration and Uncorrelated Measurement Noise")

In [ ]:
true_err_1, _, calc_err_1, _ = run_experiment(LAMBDA_UNCORRELATED, LAMBDA_UNCORRELATED, PHI, H, Q, R, X_0_0, P_0_0)

plot_errors(
    [(true_err_1[:, 0], f"True error ({M} runs)", "blue"),
     (calc_err_1[:, 0], "Calculated error $\\sigma_{x_{i,i}}$", "orange")],
    "Scenario 1: True and Calculated Errors of the Filtered Coordinate")

steady_true_1, steady_calc_1 = print_steady_errors(true_err_1[:, 0], calc_err_1[:, 0], "Scenario 1")

### Scenario 2. Effects of ignoring correlated state noise in Kalman filtering
Non-optimal filter: assumes uncorrelated state noise, but true acceleration is correlated ($\lambda = 0.1$ for $a_i$, $\lambda = 1000$ for $\eta_i$).

In [ ]:
# Scenario 2: correlated acceleration and uncorrelated measurement noise, the filter is the same as in scenario 1
run_single(LAMBDA_CORRELATED, LAMBDA_UNCORRELATED,
           "Scenario 2: Correlated Acceleration and Uncorrelated Measurement Noise")

In [ ]:
true_err_2, _, calc_err_2, _ = run_experiment(LAMBDA_CORRELATED, LAMBDA_UNCORRELATED, PHI, H, Q, R, X_0_0, P_0_0)

plot_errors(
    [(true_err_2[:, 0], f"True error ({M} runs)", "blue"),
     (calc_err_2[:, 0], "Calculated error $\\sigma_{x_{i,i}}$", "orange")],
    "Scenario 2: True and Calculated Errors of the Filtered Coordinate")

steady_true_2, steady_calc_2 = print_steady_errors(true_err_2[:, 0], calc_err_2[:, 0], "Scenario 2")

In [ ]:
# comparison with the measurement error and with the optimal filter of scenario 1
plot_errors(
    [(true_err_1[:, 0], "Scenario 1: optimal filter", "green"),
     (true_err_2[:, 0], "Scenario 2: correlated acceleration is ignored", "blue")],
    "True Errors of the Filtered Coordinate: Scenario 1 and Scenario 2",
    sigma_eta=SIGMA_ETA)

print(f"sigma_eta / true error: scenario 1 = {SIGMA_ETA / steady_true_1:.2f}, scenario 2 = {SIGMA_ETA / steady_true_2:.2f}")
print(f"true error of scenario 2 / true error of scenario 1 = {steady_true_2 / steady_true_1:.2f}")

### Scenario 3. Impact of correlated measurement noise on filter consistency
Non-optimal filter: assumes uncorrelated measurement noise, but true observations are correlated ($\lambda = 1000$ for $a_i$, $\lambda = 0.1$ for $\eta_i$).

In [ ]:
# Scenario 3: uncorrelated acceleration and correlated measurement noise
run_single(LAMBDA_UNCORRELATED, LAMBDA_CORRELATED,
           "Scenario 3: Uncorrelated Acceleration and Correlated Measurement Noise")

In [ ]:
true_err_3, _, calc_err_3, _ = run_experiment(LAMBDA_UNCORRELATED, LAMBDA_CORRELATED, PHI, H, Q, R, X_0_0, P_0_0)

plot_errors(
    [(true_err_3[:, 0], f"True error ({M} runs)", "blue"),
     (calc_err_3[:, 0], "Calculated error $\\sigma_{x_{i,i}}$", "orange")],
    "Scenario 3: True and Calculated Errors of the Filtered Coordinate")

steady_true_3, steady_calc_3 = print_steady_errors(true_err_3[:, 0], calc_err_3[:, 0], "Scenario 3")

In [ ]:
# which neglect of correlation decreases the accuracy more: of the state noise (scenario 2) or of the measurement noise (scenario 3)
plot_errors(
    [(true_err_1[:, 0], "Scenario 1: optimal filter", "green"),
     (true_err_2[:, 0], "Scenario 2: correlated acceleration is ignored", "blue"),
     (true_err_3[:, 0], "Scenario 3: correlated measurement noise is ignored", "purple")],
    "True Errors of the Filtered Coordinate: Scenarios 1, 2 and 3",
    sigma_eta=SIGMA_ETA, legend_outside=True)

print(f"sigma_eta / true error: scenario 1 = {SIGMA_ETA / steady_true_1:.2f}, "
      f"scenario 2 = {SIGMA_ETA / steady_true_2:.2f}, scenario 3 = {SIGMA_ETA / steady_true_3:.2f}")
print(f"true error / true error of scenario 1: scenario 2 = {steady_true_2 / steady_true_1:.2f}, "
      f"scenario 3 = {steady_true_3 / steady_true_1:.2f}")

### Scenario 4. Combined effects of ignoring correlated state and measurement noise
Non-optimal Kalman filter: assumed uncorrelated state and measurement noise vs. actual correlation of both noises ($\lambda = 0.1$ for $a_i$ and $\eta_i$).

In [ ]:
# Scenario 4: correlated acceleration and correlated measurement noise
run_single(LAMBDA_CORRELATED, LAMBDA_CORRELATED,
           "Scenario 4: Correlated Acceleration and Correlated Measurement Noise")

In [ ]:
true_err_4, _, calc_err_4, _ = run_experiment(LAMBDA_CORRELATED, LAMBDA_CORRELATED, PHI, H, Q, R, X_0_0, P_0_0)

plot_errors(
    [(true_err_4[:, 0], f"True error ({M} runs)", "blue"),
     (calc_err_4[:, 0], "Calculated error $\\sigma_{x_{i,i}}$", "orange")],
    "Scenario 4: True and Calculated Errors of the Filtered Coordinate")

steady_true_4, steady_calc_4 = print_steady_errors(true_err_4[:, 0], calc_err_4[:, 0], "Scenario 4")

In [ ]:
# comparison of all scenarios with each other and with the measurement error
plot_errors(
    [(true_err_1[:, 0], "Scenario 1: optimal filter", "green"),
     (true_err_2[:, 0], "Scenario 2: correlated acceleration is ignored", "blue"),
     (true_err_3[:, 0], "Scenario 3: correlated measurement noise is ignored", "purple"),
     (true_err_4[:, 0], "Scenario 4: both correlations are ignored", "black"),
     (calc_err_1[:, 0], "Calculated error $\\sigma_{x_{i,i}}$ (the same in all scenarios)", "orange")],
    "True Errors of the Filtered Coordinate in All Scenarios",
    sigma_eta=SIGMA_ETA, legend_outside=True)

# summary table of the steady-state errors
print(f"{'Scenario':<10}{'True, m':>10}{'Calculated, m':>16}{'True / calc':>14}{'sigma_eta / true':>19}")
for scenario, steady_true, steady_calc in [(1, steady_true_1, steady_calc_1), (2, steady_true_2, steady_calc_2),
                                           (3, steady_true_3, steady_calc_3), (4, steady_true_4, steady_calc_4)]:
    print(f"{scenario:<10}{steady_true:>10.2f}{steady_calc:>16.2f}{steady_true / steady_calc:>14.2f}{SIGMA_ETA / steady_true:>19.2f}")

## Part II. Optimal Kalman Filtering with Correlated State Noise
1. True trajectory with correlated acceleration ($\lambda = 0.1$) and uncorrelated measurements ($\lambda = 1000$), the same noise statistics and initial conditions as in Part I.

2. State-space model with the extended state vector $X_i = [x_i, V_i, a_i]^T$:
$$X_i = \Phi X_{i-1} + G\zeta_i, \qquad z_i = H X_i + \eta_i$$
$$\Phi = \begin{bmatrix} 1 & T & T^2/2 \\ 0 & 1 & T \\ 0 & 0 & e^{-\lambda T} \end{bmatrix}, \qquad G = \begin{bmatrix} 0 \\ 0 \\ 1 \end{bmatrix}, \qquad H = \begin{bmatrix} 1 & 0 & 0 \end{bmatrix}, \qquad Q = GG^T\sigma_a^2(1 - e^{-2\lambda T})$$

In [ ]:
# state-space model with the extended state vector X_i = [x_i, V_i, a_i]
LAMBDA_A = LAMBDA_CORRELATED
LAMBDA_ETA = LAMBDA_UNCORRELATED
VARIANCE_ZETA = VARIANCE_A * (1 - np.exp(-2 * LAMBDA_A * T))

# transition matrix Phi
PHI_EXT = np.array([[1, T, T**2 / 2],
                    [0, 1, T],
                    [0, 0, np.exp(-LAMBDA_A * T)]])

# input matrix G
G_EXT = np.array([[0],
                  [0],
                  [1]])

# observation matrix H
H_EXT = np.array([[1, 0, 0]])

# covariance matrix Q of the state noise
Q_EXT = G_EXT @ G_EXT.T * VARIANCE_ZETA

print(PHI_EXT)
print(Q_EXT)

In [ ]:
# verification of the state-space model: X_i = Phi X_{i-1} + G zeta_i must give the same trajectory as equation (4)
true_traj, true_vel, true_acc = generate_true_trajectory(LAMBDA_A)

# the noise zeta_i that was used inside equation (4)
zeta = true_acc[1:] - np.exp(-LAMBDA_A * T) * true_acc[:-1]

state = np.zeros((POINTS, 3, 1))
state[0] = np.array([[true_traj[0]],
                     [true_vel[0]],
                     [true_acc[0]]])
for i in range(1, POINTS):
    state[i] = PHI_EXT @ state[i-1] + G_EXT * zeta[i-1]

print(np.allclose(state[:, 0, 0], true_traj), np.allclose(state[:, 1, 0], true_vel), np.allclose(state[:, 2, 0], true_acc))

In [ ]:
# initial conditions for the Kalman filter algorithm
# initial filtered estimate X_{0,0}
X_0_0_EXT = np.array([[2],
                      [0],
                      [0]])

# initial filtration error covariance matrix P_{0,0}
P_0_0_EXT = np.array([[10000, 0, 0],
                      [0, 10000, 0],
                      [0, 0, 10000]])

In [ ]:
# one realization: true values and filtered estimates of the coordinate, velocity and acceleration
true_traj, true_vel, true_acc = generate_true_trajectory(LAMBDA_A)
measurements = generate_measurements(true_traj, LAMBDA_ETA)
x_pred_ext, P_pred_ext, x_filt_ext, P_filt_ext, K_ext = kalman_filter(measurements, PHI_EXT, H_EXT, Q_EXT, R, X_0_0_EXT, P_0_0_EXT)

plot_filter_results(true_traj, measurements, x_filt_ext[:, 0, 0],
                    "Optimal Filter with the Extended State Vector: Coordinate")

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(steps, true_vel, color="green", linewidth=1.5, label="True velocity $V_i$")
axes[0].plot(steps, x_filt_ext[:, 1, 0], color="blue", linewidth=1.5, label="Filtered estimate $V_{i,i}$")
axes[0].set_title("Optimal Filter with the Extended State Vector: Velocity")
axes[0].set_xlabel("Time, s")
axes[0].set_ylabel("Velocity V, m/s")
axes[0].set_ylim(np.min(true_vel) - 3, np.max(true_vel) + 3)
axes[0].grid(True, color="black", linewidth=0.3)
axes[0].legend(frameon=True)

axes[1].plot(steps, true_acc, color="green", linewidth=1.5, label="True acceleration $a_i$")
axes[1].plot(steps, x_filt_ext[:, 2, 0], color="blue", linewidth=1.5, label="Filtered estimate $a_{i,i}$")
axes[1].set_title("Optimal Filter with the Extended State Vector: Acceleration")
axes[1].set_xlabel("Time, s")
axes[1].set_ylabel("Acceleration a, m/s$^2$")
axes[1].set_ylim(np.min(true_acc) - 0.5, np.max(true_acc) + 0.5)
axes[1].grid(True, color="black", linewidth=0.3)
axes[1].legend(frameon=True)

fig.tight_layout()
plt.show()

3. True and calculated errors of the filtered and predicted estimates of $x_i$, $V_i$ and $a_i$ over $M = 500$ runs

In [ ]:
true_err_filt_ext, true_err_pred_ext, calc_err_filt_ext, calc_err_pred_ext = run_experiment(
    LAMBDA_A, LAMBDA_ETA, PHI_EXT, H_EXT, Q_EXT, R, X_0_0_EXT, P_0_0_EXT)

In [ ]:
# true and calculated errors of the filtered (left) and predicted (right) estimates for every component of the state vector
def plot_errors_ext(j, name, unit, start=5):
    steps = np.arange(POINTS * T, step = T)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(steps[start:], true_err_filt_ext[start:, j], color="blue", linewidth=1.5, label=f"True error ({M} runs)")
    axes[0].plot(steps[start:], calc_err_filt_ext[start:, j], color="orange", linewidth=1.5, label="Calculated error $\\sqrt{P_{i,i}}$")
    axes[0].set_title(f"Filtered Estimate of the {name}")
    axes[0].set_ylabel(f"Estimation error, {unit}")

    axes[1].plot(steps[start:], true_err_pred_ext[start:, j], color="blue", linewidth=1.5, label=f"True error ({M} runs)")
    axes[1].plot(steps[start:], calc_err_pred_ext[start:, j], color="orange", linewidth=1.5, label="Calculated error $\\sqrt{P_{i,i-1}}$")
    axes[1].set_title(f"Predicted Estimate of the {name}")
    axes[1].set_ylabel(f"Estimation error, {unit}")

    for ax in axes:
        ax.set_xlabel("Time, s")
        ax.grid(True, color="black", linewidth=0.3)
        ax.legend(frameon=True)
    fig.tight_layout()
    plt.show()

plot_errors_ext(0, "Coordinate $x_i$", "m")
plot_errors_ext(1, "Velocity $V_i$", "m/s")
plot_errors_ext(2, "Acceleration $a_i$", "m/s$^2$")

In [ ]:
for j, name, unit in [(0, "coordinate", "m"), (1, "velocity", "m/s"), (2, "acceleration", "m/s^2")]:
    print_steady_errors(true_err_filt_ext[:, j], calc_err_filt_ext[:, j], f"Filtered {name}", unit)
    print_steady_errors(true_err_pred_ext[:, j], calc_err_pred_ext[:, j], f"Predicted {name}", unit)

Comparison of the optimal filter with the extended state vector and the non-optimal filter of scenario 2 (the same noise statistics: correlated acceleration, uncorrelated measurements)

In [ ]:
plot_errors(
    [(true_err_2[:, 0], "Non-optimal filter $X = [x, V]$ (scenario 2): true error", "blue"),
     (calc_err_2[:, 0], "Non-optimal filter $X = [x, V]$ (scenario 2): calculated error", "orange"),
     (true_err_filt_ext[:, 0], "Optimal filter $X = [x, V, a]$: true error", "green"),
     (calc_err_filt_ext[:, 0], "Optimal filter $X = [x, V, a]$: calculated error", "black")],
    "True and Calculated Errors of the Filtered Coordinate: Non-Optimal and Optimal Filter",
    start=5, sigma_eta=SIGMA_ETA, legend_outside=True)

steady_true_ext = np.mean(true_err_filt_ext[-100:, 0])
print(f"Non-optimal filter: true error = {steady_true_2:.2f} m, sigma_eta / true = {SIGMA_ETA / steady_true_2:.2f}")
print(f"Optimal filter: true error = {steady_true_ext:.2f} m, sigma_eta / true = {SIGMA_ETA / steady_true_ext:.2f}")